In [1485]:
import numpy as np

# note here 21d means 3 weeks / 14d means 2 weeks which are 15 and 10 trading days respectively, 252 trading days in a year
available_contracts = {
    "AC": {
        "bid": {"size": 200, "price": 49.975},
        "ask": {"size": 200, "price": 50.025},
        "type": "underlying",
    },
    "AC_50_P": {
        "bid": {"size": 50, "price": 12},
        "ask": {"size": 50, "price": 12.05},
        "type": "put",
        "strike": 50,
        "expiry": '21d',
    },
    "AC_50_C": {
        "bid": {"size": 50, "price": 12},
        "ask": {"size": 50, "price": 12.05},
        "type": "call",
        "strike": 50,
        "expiry": '21d',
    },
    "AC_35_P": {
        "bid": {"size": 50, "price": 4.33},
        "ask": {"size": 50, "price": 4.35},
        "type": "put",
        "strike": 35,
        "expiry": '21d',
    },
    "AC_40_P": {
        "bid": {"size": 50, "price": 6.5},
        "ask": {"size": 50, "price": 6.55},
        "type": "put",
        "strike": 40,
        "expiry": '21d',
    },
    "AC_45_P": {
        "bid": {"size": 50, "price": 9.05},
        "ask": {"size": 50, "price": 9.1},
        "type": "put",
        "strike": 45,
        "expiry": '21d',
    },
    "AC_60_C": {
        "bid": {"size": 50, "price": 8.8},
        "ask": {"size": 50, "price": 8.85},
        "type": "call",
        "strike": 60,
        "expiry": '21d',
    },
    "AC_50_P_2": {
        "bid": {"size": 50, "price": 9.7},
        "ask": {"size": 50, "price": 9.75},
        "type": "put",
        "strike": 50,
        "expiry": '14d',
    },
    "AC_50_C_2": {
        "bid": {"size": 50, "price": 9.7},
        "ask": {"size": 50, "price": 9.75},
        "type": "call",
        "strike": 50,
        "expiry": '14d',
    },
    "AC_50_C0": {
        "bid": {"size": 50, "price": 22.2},
        "ask": {"size": 50, "price": 22.3},
        "type": "chooser",
        "strike": 50,
        "expiry": '14d/21d',
    },
    "AC_40_BP": {
        "bid": {"size": 50, "price": 5},
        "ask": {"size": 50, "price": 5.1},
        "type": "binary_put",
        "strike": 40,
        "payoff": 10, # payoff per contract
        "expiry": '21d',
    },
    "AC_45_KO": {
        "bid": {"size": 500, "price": 0.15},
        "ask": {"size": 500, "price": 0.175},
        "type": "knockout_put",
        "strike": 45,
        "barrier_price": 35,
        "expiry": '21d',
    },
}

In [1486]:
# GBM sim

# Parameters
S0 = 50
sigma = 2.51  # 251% annualized
TRADING_DAYS_PER_YEAR = 252
STEPS_PER_DAY = 4
STEPS_PER_YEAR = TRADING_DAYS_PER_YEAR * STEPS_PER_DAY
dt = 1 / STEPS_PER_YEAR

def simulate_gbm(S0, sigma, n_steps, n_sims, seed=None):
    """
    Simulates GBM paths.
    Returns array of shape (n_sims, n_steps + 1)
    including S0 at index 0.
    """
    if seed is not None:
        np.random.seed(seed)

    Z = np.random.normal(0, 1, (n_sims, n_steps))
    log_returns = (- 0.5 * sigma**2 * dt) + sigma * np.sqrt(dt) * Z
    log_paths = np.cumsum(log_returns, axis=1)
    paths = S0 * np.exp(np.insert(log_paths, 0, 0, axis=1))

    return paths

print(simulate_gbm(S0, sigma, n_steps=60, n_sims=1))

[[50.         47.8324045  52.9619247  50.59125572 47.42855626 45.34068225
  46.62050308 43.53617575 46.60017834 46.42000866 47.00805261 48.16216224
  44.19625399 47.45808153 37.80750919 34.45699715 37.95238322 34.84648568
  30.45413869 31.1407465  27.44802    26.03360998 27.46872056 25.02761459
  25.75007416 26.28832923 25.83353905 25.30522467 25.93732034 23.91546194
  24.04387683 21.57662579 20.85826487 21.80327955 20.69191815 20.257985
  21.29233518 20.27369098 18.74739722 19.66887989 22.23542656 22.45613498
  20.34820451 22.0854013  23.13138035 22.97843261 21.90981303 23.288292
  24.42361065 24.9908902  21.45113875 19.56659249 20.61539861 23.56027455
  27.15068776 24.14440959 22.73673881 22.72890954 22.77150231 22.57047834
  21.93543901]]


In [1487]:
# payoff functions
def payoff_vanilla_call(paths, strike, expiry_steps):
    S_T = paths[:, expiry_steps]
    return np.maximum(S_T - strike, 0)

def payoff_vanilla_put(paths, strike, expiry_steps):
    S_T = paths[:, expiry_steps]
    return np.maximum(strike - S_T, 0)

def payoff_chooser(paths, strike, choice_steps, expiry_steps):
    # At choice point, pick whichever is ITM
    S_choice = paths[:, choice_steps]
    S_expiry = paths[:, expiry_steps]

    # If above strike at choice date → call, else → put
    is_call = S_choice >= strike
    call_payoff = np.maximum(S_expiry - strike, 0)
    put_payoff = np.maximum(strike - S_expiry, 0)

    return np.where(is_call, call_payoff, put_payoff)

def payoff_binary_put(paths, strike, expiry_steps, payout):
    S_T = paths[:, expiry_steps]
    return np.where(S_T < strike, payout, 0)

def payoff_knockout_put(paths, strike, barrier, expiry_steps):
    S_T = paths[:, expiry_steps]
    # knocked out if ANY step strictly below barrier
    knocked_out = np.any(paths[:, 1:expiry_steps+1] < barrier, axis=1)
    vanilla_payoff = np.maximum(strike - S_T, 0)
    return np.where(knocked_out, 0, vanilla_payoff)

In [1488]:
def price_all_contracts(available_contracts, n_sims=1000000, seed=42):
    """
    Run MC simulation and price all contracts.
    Use large n_sims for accurate fair values,
    then compare against market mid prices.
    """
    # Generate paths once - reuse for all contracts
    paths_60 = simulate_gbm(S0, sigma, 60, n_sims, seed=None)
    paths_40 = paths_60[:, :41]  # subset for 2-week expiry

    results = {}

    for symbol, contract in available_contracts.items():
        mid = (contract['bid']['price'] + contract['ask']['price']) / 2

        if contract['type'] == 'underlying':
            continue

        elif contract['type'] == 'call':
            steps = 40 if contract['expiry'] == '14d' else 60
            payoffs = payoff_vanilla_call(paths_60, contract['strike'], steps)

        elif contract['type'] == 'put':
            steps = 40 if contract['expiry'] == '14d' else 60
            payoffs = payoff_vanilla_put(paths_60, contract['strike'], steps)

        elif contract['type'] == 'chooser':
            payoffs = payoff_chooser(paths_60, contract['strike'], 40, 60)

        elif contract['type'] == 'binary_put':
            payoffs = payoff_binary_put(paths_60, contract['strike'], 60, contract['payoff'])

        elif contract['type'] == 'knockout_put':
            payoffs = payoff_knockout_put(paths_60, contract['strike'],
                                         contract['barrier_price'], 60)

        fair_value = np.mean(payoffs)
        std = np.std(payoffs)
        mispricing = fair_value - mid

        results[symbol] = {
            'fair_value': fair_value,
            'market_mid': mid,
            'mispricing': mispricing,  # positive = market underpriced = BUY
            'std': std,
            'bid': contract['bid']['price'],
            'ask': contract['ask']['price'],
            'max_size': contract['bid']['size'],
        }

    return results

def print_results(results):
    print(f"{'Symbol':<15} {'Fair Value':>10} {'Market Mid':>10} {'Mispricing':>10} {'Std':>10} {'Signal':>8}")
    print("-" * 65)
    for symbol, r in results.items():
        signal = "BUY" if r['mispricing'] > 0 else "SELL"
        print(f"{symbol:<15} {r['fair_value']:>10.4f} {r['market_mid']:>10.4f} "
              f"{r['mispricing']:>10.4f} {r['std']:>10.4f} {signal:>8}")

results = price_all_contracts(available_contracts)
print_results(results)

Symbol          Fair Value Market Mid Mispricing        Std   Signal
-----------------------------------------------------------------
AC_50_P            12.0343    12.0250     0.0093    12.6088      BUY
AC_50_C            12.0142    12.0250    -0.0108    26.2701     SELL
AC_35_P             4.3359     4.3400    -0.0041     6.9318     SELL
AC_40_P             6.5120     6.5250    -0.0130     8.8503     SELL
AC_45_P             9.0939     9.0750     0.0189    10.7585      BUY
AC_60_C             8.7820     8.8250    -0.0430    23.4766     SELL
AC_50_P_2           9.8778     9.7250     0.1528    10.9712      BUY
AC_50_C_2           9.8755     9.7250     0.1505    19.9180      BUY
AC_50_C0           21.8966    22.2500    -0.3534    24.8032     SELL
AC_40_BP            4.7727     5.0500    -0.2773     4.9948     SELL
AC_45_KO            0.2064     0.1625     0.0439     1.0873      BUY


In [1489]:
def compute_pnl(results, trades):
    """
    trades: dict of {symbol: signed_quantity}
    positive = buy, negative = sell
    """
    CONTRACT_SIZE = 3000

    total_expected_pnl = 0
    total_variance = 0

    print(f"{'Symbol':<15} {'Qty':>6} {'Entry':>8} {'Fair Value':>10} {'Edge/Unit':>10} {'Expected PnL':>12}")
    print("-" * 70)

    for symbol, qty in trades.items():
        r = results[symbol]
        contract = available_contracts[symbol]

        # Entry price: buy at ask, sell at bid
        entry = contract['ask']['price'] if qty > 0 else contract['bid']['price']

        edge_per_unit = (r['fair_value'] - entry) * np.sign(qty)
        expected_pnl = edge_per_unit * abs(qty) * CONTRACT_SIZE
        variance = (r['std'] ** 2) * (qty ** 2) * (CONTRACT_SIZE ** 2)

        total_expected_pnl += expected_pnl
        total_variance += variance  # note: ignores covariance between legs

        print(f"{symbol:<15} {qty:>6} {entry:>8.4f} {r['fair_value']:>10.4f} "
              f"{edge_per_unit:>10.4f} {expected_pnl:>12.2f}")

    total_std = np.sqrt(total_variance)
    print("-" * 70)
    print(f"{'TOTAL':<15} {'':>6} {'':>8} {'':>10} {'':>10} {total_expected_pnl:>12.2f}")
    print(f"{'Std Dev':<15} {'':>6} {'':>8} {'':>10} {'':>10} {total_std:>12.2f}")
    print(f"{'Sharpe':<15} {'':>6} {'':>8} {'':>10} {'':>10} "
          f"{total_expected_pnl/total_std:>12.4f}")

# Chooser arb: sell chooser, buy call + put replicating portfolio
# Binary put: sell (overpriced)
# Knockout put: buy (underpriced)

trades = {
    "AC_50_C0": -50,   # sell chooser at max size
    #"AC_50_C": +50,    # buy call leg
    "AC_50_P_2": +50,  # buy put leg
    "AC_50_C_2": +50,  # buy 2 week call (underpriced)
    "AC_40_BP": -50,   # sell binary put
    "AC_45_KO": +500,  # buy knockout put
}

compute_pnl(results, trades)

Symbol             Qty    Entry Fair Value  Edge/Unit Expected PnL
----------------------------------------------------------------------
AC_50_C0           -50  22.2000    21.8966     0.3034     45516.91
AC_50_P_2           50   9.7500     9.8778     0.1278     19169.72
AC_50_C_2           50   9.7500     9.8755     0.1255     18818.25
AC_40_BP           -50   5.0000     4.7727     0.2273     34095.00
AC_45_KO           500   0.1750     0.2064     0.0314     47133.16
----------------------------------------------------------------------
TOTAL                                                    164733.05
Std Dev                                                 5357057.96
Sharpe                                                      0.0308


In [1490]:
def live_score(trades, n_score_sims=100, n_repeats=1):
    """
    Simulates the competition scoring process.
    n_score_sims=100: exactly as competition runs it
    n_repeats: how many times we repeat to get score distribution
    """
    CONTRACT_SIZE = 3000
    scores = []

    for i in range(n_repeats):
        # Each repeat is one possible competition outcome
        paths = simulate_gbm(S0, sigma, 60, n_score_sims)
        total_pnl = 0

        for symbol, qty in trades.items():
            contract = available_contracts[symbol]
            entry = contract['ask']['price'] if qty > 0 else contract['bid']['price']

            if contract['type'] == 'call':
                steps = 40 if contract['expiry'] == '14d' else 60
                payoffs = payoff_vanilla_call(paths, contract['strike'], steps)
            elif contract['type'] == 'put':
                steps = 40 if contract['expiry'] == '14d' else 60
                payoffs = payoff_vanilla_put(paths, contract['strike'], steps)
            elif contract['type'] == 'chooser':
                payoffs = payoff_chooser(paths, contract['strike'], 40, 60)
            elif contract['type'] == 'binary_put':
                payoffs = payoff_binary_put(paths, contract['strike'], 60, contract['payoff'])
            elif contract['type'] == 'knockout_put':
                payoffs = payoff_knockout_put(paths, contract['strike'],
                                             contract['barrier_price'], 60)

            # per path PnL, shape (n_score_sims,)
            per_path_pnl = (payoffs - entry) * qty * CONTRACT_SIZE
            total_pnl += per_path_pnl  # accumulate across contracts

        scores.append(np.mean(total_pnl))

    scores = np.array(scores)
    return scores

def print_score_distribution(scores):
    print(f"Expected Score:      {np.mean(scores):>12.2f}")
    print(f"Std Dev:             {np.std(scores):>12.2f}")
    print(f"5th percentile:      {np.percentile(scores, 5):>12.2f}")
    print(f"25th percentile:     {np.percentile(scores, 25):>12.2f}")
    print(f"Median:              {np.percentile(scores, 50):>12.2f}")
    print(f"75th percentile:     {np.percentile(scores, 75):>12.2f}")
    print(f"95th percentile:     {np.percentile(scores, 95):>12.2f}")
    print(f"Prob positive:       {np.mean(scores > 0):>12.2%}")

score = live_score(trades)
print_score_distribution(score)

Expected Score:         -12703.12
Std Dev:                     0.00
5th percentile:         -12703.12
25th percentile:        -12703.12
Median:                 -12703.12
75th percentile:        -12703.12
95th percentile:        -12703.12
Prob positive:              0.00%


In [1491]:
from scipy.stats import norm

def black_scholes(S, K, T, sigma, option_type='call'):
    """
    Black-Scholes pricer for European options.
    S: spot price
    K: strike
    T: time to expiry in years
    sigma: annualized volatility
    option_type: 'call' or 'put'
    r=0 assumed throughout (zero risk-free rate)
    """
    d1 = (np.log(S / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = S * norm.cdf(d1) - K * norm.cdf(d2)
    elif option_type == 'put':
        price = K * norm.cdf(-d2) - S * norm.cdf(-d1)

    return price

def weeks_to_years(weeks):
    return (weeks * 5) / 252

# Price all vanillas
vanilla_contracts = {k: v for k, v in available_contracts.items()
                     if v['type'] in ('call', 'put')}

print(f"{'Symbol':<15} {'BS Price':>10} {'MC Price':>10} {'Market Mid':>10} {'BS vs Mid':>10}")
print("-" * 60)

for symbol, contract in vanilla_contracts.items():
    weeks = 2 if contract['expiry'] == '14d' else 3
    T = weeks_to_years(weeks)

    bs_price = black_scholes(S0, contract['strike'], T, sigma, contract['type'])
    mc_price = results[symbol]['fair_value']
    mid = (contract['bid']['price'] + contract['ask']['price']) / 2

    print(f"{symbol:<15} {bs_price:>10.4f} {mc_price:>10.4f} {mid:>10.4f} {bs_price - mid:>10.4f}")

Symbol            BS Price   MC Price Market Mid  BS vs Mid
------------------------------------------------------------
AC_50_P            12.0269    12.0343    12.0250     0.0019
AC_50_C            12.0269    12.0142    12.0250     0.0019
AC_35_P             4.3361     4.3359     4.3400    -0.0039
AC_40_P             6.5095     6.5120     6.5250    -0.0155
AC_45_P             9.0889     9.0939     9.0750     0.0139
AC_60_C             8.7918     8.7820     8.8250    -0.0332
AC_50_P_2           9.8707     9.8778     9.7250     0.1457
AC_50_C_2           9.8707     9.8755     9.7250     0.1457
